# Exemplo de Execução do Pipeline (CNN 2D)

Este notebook demonstra o uso passo a passo dos componentes do pipeline de Machine Learning.

In [1]:
import os
import sys
import torch
import numpy as np
from sklearn.model_selection import train_test_split

# Em notebooks, __file__ não existe por padrão. Usamos o diretório atual para encontrar a raiz do projeto.
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

from ai.loader.loader import DataLoader
from ai.label.label_generator import label_from_path
from ai.preprocess.cnn2d import PreprocessCNN2D
from ai.models.cnn2d import ModelCNN2D
from ai.trainer.trainer import ModelTrainer
from ai.evaluation.monitor import ModelMonitor
from ai.evaluation.summary import ModelSummary
from ai.preprocess.balancer import DataBalancer

## 1. Carregamento dos Dados

Vamos usar a classe `DataLoader` para buscar os arquivos parquet e carregá-los em um DataFrame do Pandas.

In [2]:
# Ajuste o data_path para apontar para seus arquivos reais de dados.
# Se não passar nada, o loader buscará por padrão na pasta data/parquet/
data_path = os.path.join(PROJECT_ROOT, "data", "parquet", "**", "*.parquet")

loader = DataLoader(data_path=data_path, max_files=2) # Limitando a 1 arquivo para o exemplo rodar rápido
df = loader.execute()

if df is None or df.empty:
    print("Erro: Nenhum dado foi carregado.")

print("\n-> Geração de Labels...")
df["label"] = df["file_path"].apply(label_from_path)
print("DataFrame\n")
print("-----CAMINHO-----")
print(df["file_path"][100])
print("-----LABEL-----")
print(df["label"][100])
print("-----CAMINHO 2-----")
print(df["file_path"][7500])
print("-----LABEL-----")
print(df["label"][7500])

📥 Loading Parquets: 100%|██████████| 8/8 [00:00<00:00, 31.27file/s]


-> Geração de Labels...
DataFrame

-----CAMINHO-----
/home/joao.gomes/LPS/cern/data/parquet/mc25_13TeV.20260104.physics_Main.JF17.100k.r1.JF17.parquet/JF17.0.parquet
-----LABEL-----
0
-----CAMINHO 2-----
/home/joao.gomes/LPS/cern/data/parquet/mc25_13TeV.20251215.physics_Main.Zee.500k.r1.Zee.parquet/Zee.1.parquet
-----LABEL-----
1


## 2. Pré-processamento

O pré-processador formata as variáveis em tensores adequados para a CNN (imagens 2D com múltiplos canais).

In [3]:
preprocessor = PreprocessCNN2D()

X = preprocessor.transform(df)
Y = preprocessor.get_labels(df, label_col='label')
    
print(f"Formato de X (Features): {X.shape}")
print(f"Formato de Y (Labels): {Y.shape}")

print("Entrada:",X)
print("Labels:",Y)

Processing Samples: 100%|██████████| 15100/15100 [00:00<00:00, 27024.59it/s]

Formato de X (Features): (15100, 7, 7, 15)
Formato de Y (Labels): (15100,)
Entrada: [[[[0. 0. 0. ... 0. 0. 0.]
   [0. 0. 0. ... 0. 0. 0.]
   [0. 0. 0. ... 0. 0. 0.]
   ...
   [0. 0. 0. ... 0. 0. 0.]
   [0. 0. 0. ... 0. 0. 0.]
   [0. 0. 0. ... 0. 0. 0.]]

  [[0. 0. 0. ... 0. 0. 0.]
   [0. 0. 0. ... 0. 0. 0.]
   [0. 0. 0. ... 0. 0. 0.]
   ...
   [0. 0. 0. ... 0. 0. 0.]
   [0. 0. 0. ... 0. 0. 0.]
   [0. 0. 0. ... 0. 0. 0.]]

  [[0. 0. 0. ... 0. 0. 0.]
   [0. 0. 0. ... 0. 0. 0.]
   [0. 0. 0. ... 0. 0. 0.]
   ...
   [0. 0. 0. ... 0. 0. 0.]
   [0. 0. 0. ... 0. 0. 0.]
   [0. 0. 0. ... 0. 0. 0.]]

  ...

  [[0. 0. 0. ... 0. 0. 0.]
   [0. 0. 0. ... 0. 0. 0.]
   [0. 0. 0. ... 0. 0. 0.]
   ...
   [0. 0. 0. ... 0. 0. 0.]
   [0. 0. 0. ... 0. 0. 0.]
   [0. 0. 0. ... 0. 0. 0.]]

  [[0. 0. 0. ... 0. 0. 0.]
   [0. 0. 0. ... 0. 0. 0.]
   [0. 0. 0. ... 0. 0. 0.]
   ...
   [0. 0. 0. ... 0. 0. 0.]
   [0. 0. 0. ... 0. 0. 0.]
   [0. 0. 0. ... 0. 0. 0.]]

  [[0. 0. 0. ... 0. 0. 0.]
   [0. 0. 0. ... 0. 0. 0.]


## 3. Balanceamento dos Dados

Realiza o balanceamento dos dados utilizando Undersampling

In [4]:
data_balancer = DataBalancer()
X_balanced, Y_balanced = data_balancer.apply(X, Y)

## 4. Divisão de Dados (Train / Test)

Separamos um conjunto de teste isolado.

In [5]:
X_train, X_test, Y_train, Y_test = train_test_split(X_balanced, Y_balanced, test_size=0.15, random_state=42, shuffle = True)
print(f"Treinamento: {X_train.shape[0]} amostras")
print(f"Teste Isolado: {X_test.shape[0]} amostras")
print(Y_train)
print(Y_test)

uns_train = np.sum(Y_train == 1.0)
zeros_train = np.sum(Y_train == 0.0)

uns_test = np.sum(Y_test == 1.0)
zeros_test = np.sum(Y_test == 0.0)

print("Quantidade de 1 no conjunto de teste:",uns_test)
print("Quantidade de 1 no conjunto de treino:",uns_train)
print("Quantidade de 0 no conjunto de teste:",zeros_test)
print("Quantidade de 0 no conjunto de treino:",zeros_train)

Treinamento: 1089 amostras
Teste Isolado: 193 amostras
[1. 1. 0. ... 1. 1. 0.]
[1. 0. 0. 0. 1. 0. 0. 0. 1. 1. 0. 0. 1. 1. 1. 1. 1. 1. 0. 0. 1. 0. 0. 1.
 1. 1. 1. 1. 0. 1. 1. 0. 0. 0. 0. 0. 0. 1. 1. 1. 0. 0. 0. 0. 1. 1. 0. 0.
 1. 1. 1. 1. 0. 0. 0. 1. 0. 1. 1. 1. 1. 1. 1. 1. 0. 1. 0. 1. 0. 0. 1. 1.
 1. 0. 0. 0. 0. 1. 1. 0. 1. 1. 1. 0. 1. 1. 0. 1. 1. 1. 1. 0. 0. 1. 1. 1.
 0. 0. 1. 1. 1. 0. 0. 1. 0. 0. 1. 0. 0. 0. 1. 0. 0. 0. 0. 1. 1. 1. 1. 0.
 1. 0. 0. 1. 1. 1. 0. 0. 1. 1. 0. 0. 1. 0. 0. 0. 0. 1. 0. 0. 1. 1. 1. 0.
 1. 1. 1. 0. 1. 1. 0. 0. 0. 1. 0. 0. 0. 1. 0. 1. 0. 1. 0. 1. 1. 1. 0. 1.
 1. 1. 0. 1. 1. 1. 1. 1. 1. 1. 0. 1. 1. 0. 0. 1. 0. 0. 1. 0. 1. 0. 0. 0.
 1.]
Quantidade de 1 no conjunto de teste: 103
Quantidade de 1 no conjunto de treino: 538
Quantidade de 0 no conjunto de teste: 90
Quantidade de 0 no conjunto de treino: 551


## 5. Treinamento

Configuramos o treinador (`ModelTrainer`) que lidará com o ciclo de vida do PyTorch Lightning. Podemos usar Holdout Simples ou K-Fold.

In [ ]:
results_dir = os.path.join(PROJECT_ROOT, "results", "CNN2D")

trainer = ModelTrainer(
    max_epochs=10, 
    batch_size=64,
    patience=5,
    num_workers=0,
    log_dir=os.path.join(results_dir, "lightning_logs")
)

model_kwargs = {'learning_rate': 0.001}

fold_trainers, fold_models, fold_loss_callbacks = trainer.fit_kfold(
    ModelCNN2D, model_kwargs, X_train, Y_train, 
    n_splits=2, target_fold=None
)

    
# Treino Simples (Holdout)
print("\nIniciando treinamento simples...")
model = ModelCNN2D(learning_rate=0.001)
trained_trainer = trainer.fit(model, X_train, Y_train)

/home/joao.gomes/LPS/cern/neuralnet-env/lib/python3.13/site-packages/lightning_fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /home/joao.gomes/LPS/cern/neuralnet-env/lib/python3. ...
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
/home/joao.gomes/LPS/cern/neuralnet-env/lib/python3.13/site-packages/pytorch_lightning/trainer/connectors/logger_connector/logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `pytorch_lightning` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default
💡 Tip: For seam

Sanity Checking DataLoader 0:   0%|          | 0/2 [00:00<?, ?it/s]

/home/joao.gomes/LPS/cern/neuralnet-env/lib/python3.13/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/home/joao.gomes/LPS/cern/neuralnet-env/lib/python3.13/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.
/home/joao.gomes/LPS/cern/neuralnet-env/lib/python3.13/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.
/home/joao.gomes/LPS/cern/neuralnet-env/lib/python3.13/site-packages/pytorch_lightning/loops/fit_loop.py:321: The

Epoch 0: 100%|██████████| 12/12 [00:00<00:00, 82.63it/s, v_num=7, train_loss_step=0.592, val_loss=0.664, val_acc=0.504, val_auc=0.796, train_loss_epoch=0.670, train_acc=0.577]

Metric val_loss improved. New best score: 0.664


...

Metric val_loss improved by 0.042 >= min_delta = 0.0. New best score: 0.382


Epoch 9: 100%|██████████| 12/12 [00:00<00:00, 91.09it/s, v_num=7, train_loss_step=0.197, val_loss=0.434, val_acc=0.818, val_auc=0.920, train_loss_epoch=0.253, train_acc=0.897] 

`Trainer.fit` stopped: `max_epochs=10` reached.


Epoch 9: 100%|██████████| 12/12 [00:00<00:00, 89.00it/s, v_num=7, train_loss_step=0.197, val_loss=0.434, val_acc=0.818, val_auc=0.920, train_loss_epoch=0.253, train_acc=0.897]


ValueError: too many values to unpack (expected 2)

## 6. Avaliação

Vamos avaliar o modelo treinado em nosso Test Set (dados isolados).

In [ ]:
results_dir=os.path.join("results", "CNN_2D_Test")
summary = ModelSummary(output_dir=os.path.join(results_dir, "metrics"))
monitor = ModelMonitor(output_dir=os.path.join(results_dir, "plots"))
model.eval()
with torch.no_grad():
    X_tensor = torch.as_tensor(X_test, dtype=torch.float32)
    logits = model(X_tensor)
    y_prob = torch.sigmoid(logits).numpy().flatten()

y_true = Y_test.flatten()
y_pred = (y_prob >= 0.8).astype(int)
        
file_suffix = f"fold_{fold_idx}"
summary.save_metrics(y_true, y_prob, threshold=0.8, filename=f"test_metrics{file_suffix}.csv")
monitor.plot_roc_curve(y_true, y_prob, filename=f"roc_curve{file_suffix}.pdf")
monitor.plot_confusion_matrix(y_true, y_pred, filename=f"confusion_matrix{file_suffix}.pdf")
monitor.plot_loss(loss_callback.train_loss, loss_callback.val_loss, filename=f"loss_curve{file_suffix}.pdf")